In [ ]:
import numpy as np
import rasterio
from rasterio.windows import Window
from rasterio.mask import mask
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json
from typing import Tuple, List, Dict
from tqdm import tqdm
import os
from matplotlib.colors import ListedColormap, BoundaryNorm
from scipy.ndimage import zoom
import numpy as np
import numpy as np
import random
import json
from pathlib import Path
from typing import Dict, List, Tuple
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np
from typing import Dict, Tuple
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import numpy as np
from pathlib import Path

from data_loader import ( load_dem, load_infiltration_map, load_landuse_map, get_raster_stats, print_raster_stats,visualize_raster, visualize_flood_maps_grid, 
                         load_all_rainfall_scenarios, get_rainfall_stats, load_multiple_map, load_all_flood_maps, visualize_multiple_grid)
from data_preprocessing import (normalize_dem, normalize_infiltration, normalize_landuse, handle_nodata )
from flood_maps import (FLOOD_CLASSES, reconstruct_map_from_patches, categorize_all_flood_maps_patch, visualize_all_categorized_maps)
from vit_architecture import ViTFloodClassifier, get_model_summary
from model_config import (create_small_model, update_dataset_info)
from training import train_kfold, calculate_class_weights
from testing import (evaluate_test_scenarios, plot_confusion_matrix, plot_scenario_comparison, reconstruct_test_map_from_patches)

RANDOM_SEED = 42
NUM_TRAIN = 15
NUM_TEST = 5
NUM_SCENARIO = 50
PATCH_SIZE = 4

# Hyperparameters
BATCH_SIZE = 64
NUM_WORKERS = 4     
PIN_MEMORY = False 

colors = [FLOOD_CLASSES[i]['color'] for i in range(5)]
cmap = ListedColormap(colors)
bounds = [-0.5, 0.5, 1.5, 2.5, 3.5, 4.5]
norm = BoundaryNorm(bounds, cmap.N)



In [ ]:
repo_root = os.getcwd()
gmm_path = os.path.join(repo_root, "COP-30m-GMM")
manila_path = os.path.join(repo_root, "COP-30m-ManilaOnly")
rs_path = os.path.join(repo_root, "rc_randomized_mm_hr_scenarios")
train_rs_path = os.path.join(rs_path, "train")
test_rs_path = os.path.join(rs_path, "validation")
drainage_path = os.path.join(repo_root, "drainage")
soil_path = os.path.join(repo_root, "soil_types")
building_path = os.path.join(repo_root, "building_info")


# Loading data

In [ ]:
# Load DEM
print("\n--- Loading DEM ---")
dem_path = os.path.join(manila_path, "by_box", "manila_bbox_dem_cop.tif")
if os.path.exists(dem_path):
    dem_data, dem_meta = load_dem(dem_path)
    dem_stats = get_raster_stats(dem_data, "DEM")
    print_raster_stats(dem_stats)
else:
    print(f"WARNING: DEM file not found at {dem_path}")
    dem_data = None


In [ ]:
fig1 = visualize_raster(dem_data, title="DEM - Manila", cmap='terrain')

In [ ]:
# Load Infiltration Map
print("\n--- Loading Infiltration Map ---")
infilt_path = os.path.join(manila_path, "by_box", "GM_Infilt_Manila_box.tif")
if os.path.exists(infilt_path):
    infilt_data, infilt_meta = load_infiltration_map(infilt_path)
    infilt_stats = get_raster_stats(infilt_data, "Infiltration Map")
    print_raster_stats(infilt_stats)
else:
    print(f"WARNING: Infiltration file not found at {infilt_path}")
    infilt_data = None

In [ ]:
fig2 = visualize_raster(infilt_data, title="Infiltration Map", cmap='YlGnBu')

In [ ]:
# Load Landuse Map
print("\n--- Loading Landuse Map ---")
landuse_path = os.path.join(manila_path, "by_box", "GM_LU_Manila_box.tif")
if os.path.exists(landuse_path):
    landuse_data, landuse_meta = load_landuse_map(landuse_path)
    landuse_stats = get_raster_stats(landuse_data, "Landuse Map")
    print_raster_stats(landuse_stats)
else:
    print(f"WARNING: Landuse file not found at {landuse_path}")
    landuse_data = None


In [ ]:
fig3 = visualize_raster(landuse_data, title="Landuse Map", cmap='tab20')

In [ ]:
print("\n--- Loading Drainage ---")
drainage_path = os.path.join(drainage_path)

if os.path.exists(drainage_path):
    drainage_data, drainage_meta = load_multiple_map(drainage_path)
    
    # Access individual layers by filename
    for layer_name, layer_data in drainage_data.items():
        print(f"\n  Layer: {layer_name}")
        stats = get_raster_stats(layer_data, layer_name)
        print_raster_stats(stats)
else:
    print(f"WARNING: Drainage folder not found at {drainage_path}")
    drainage_data = None

In [ ]:
fig = visualize_multiple_grid(data=drainage_data, figsize=(20, 4), cmap='Blues')

In [ ]:
print("\n--- Loading Soil Types ---")
soil_types = os.path.join(soil_path)

if os.path.exists(soil_types):
    soil_data, soil_meta = load_multiple_map(soil_types)
    
    for layer_name, layer_data in soil_data.items():
        print(f"\n  Layer: {layer_name}")
        stats = get_raster_stats(layer_data, layer_name)
        print_raster_stats(stats)
else:
    print(f"WARNING: Soil folder not found at {soil_types}")
    soil_data = None

In [ ]:
fig = visualize_multiple_grid(data=soil_data, figsize=(20, 4), cmap='Blues')

In [ ]:
# Load all flood maps
print("\n--- Loading Flood Maps (Ground Truth) ---")
fm_path = os.path.join(manila_path, "by_box")
flood_maps, flood_metadata = load_all_flood_maps(fm_path, num_scenarios=NUM_SCENARIO)

if len(flood_maps) > 0:
    print(f"\nFlood maps successfully loaded: {len(flood_maps)}")
    
    if len(flood_maps) >= NUM_SCENARIO:
        flood_stats = get_raster_stats(flood_maps[-1], "Flood Map RS20")
else:
    print("WARNING: No flood maps loaded!")

In [ ]:
print_raster_stats(flood_stats)

In [ ]:
fig4 = visualize_flood_maps_grid(flood_maps=flood_maps,scenario_ids=list(range(1, 51)),figsize=(25, 20),ncols=5,cmap='Blues')
#plt.show()

In [ ]:
from RC_randomizer import split_rs_dataset #, split_rs_dataset_three_way

source = "mm_hr_scenarios"
output = "rc_randomized_mm_hr_scenarios"

split_rs_dataset(source_dir=source, output_dir=output)

In [ ]:
print("LOADING TRAINING RAINFALL SCENARIOS")
# Load rainfall scenarios
train_rainfall_scenarios = load_all_rainfall_scenarios(train_rs_path)

if len(train_rainfall_scenarios) > 0:
    print(f"\nRainfall scenarios successfully loaded: {len(train_rainfall_scenarios)}")
    
    if len(train_rainfall_scenarios) >= NUM_SCENARIO:
        rain_stats = get_rainfall_stats(train_rainfall_scenarios[-1], scenario_id=50)
else:
    print("WARNING: No rainfall scenarios loaded!")

In [ ]:
print("LOADING VALIDATION RAINFALL SCENARIOS")
# Load rainfall scenarios
test_rainfall_scenarios = load_all_rainfall_scenarios(test_rs_path)

if len(test_rainfall_scenarios) > 0:
    print(f"\nRainfall scenarios successfully loaded: {len(test_rainfall_scenarios)}")
    
    if len(test_rainfall_scenarios) >= NUM_SCENARIO:
        rain_stats = get_rainfall_stats(test_rainfall_scenarios[-1], scenario_id=50)
else:
    print("WARNING: No rainfall scenarios loaded!")

# Data Preprocessing

## Handle NoData

In [ ]:
print("\nProcessing DEM...")
dem_clean = handle_nodata(
    dem_data, 
    nodata_value=dem_meta.get('nodata'),
    fill_method='mean'  # Options: 'mean', 'median', 'zero', 'interpolate'
)
# Verify
# dem_clean_stats = get_raster_stats(dem_clean, "DEM (NoData Handled)")
# print_raster_stats(dem_clean_stats)

In [ ]:
# 2. Handle NoData in Infiltration
print("\nProcessing Infiltration...")
infilt_clean = handle_nodata(
    infilt_data,
    nodata_value=infilt_meta.get('nodata'),
    fill_method='mean'  # Options: 'mean', 'median', 'zero', 'interpolate'
)

# Verify
# infilt_clean_stats = get_raster_stats(infilt_clean, "Infiltration (NoData Handled)")
# print_raster_stats(infilt_clean_stats)

In [ ]:
# 3. Handle NoData in Landuse
print("\nProcessing Landuse...")
landuse_clean = handle_nodata(
    landuse_data,
    nodata_value=landuse_meta.get('nodata'),
    fill_method='interpolate'  # Interpolate is better for categorical data
)

# Verify
# landuse_clean_stats = get_raster_stats(landuse_clean, "Landuse (NoData Handled)")
# print_raster_stats(landuse_clean_stats)

In [ ]:
# 4. Handle NoData in Drainage
drainage_clean = {}
for layer_name, layer_data in drainage_data.items():
    drainage_clean[layer_name] = handle_nodata(
        layer_data,
        fill_method='mean',
        name=layer_name
    )


In [ ]:
# 5. Handle NoData in Soil Types
soil_clean = {}
for layer_name, layer_data in soil_data.items():
    soil_clean[layer_name] = handle_nodata(
        layer_data,
        fill_method='mean',
        name=layer_name
    )


## Normalization

In [ ]:
# Normalize DEM
print("\nNormalizing DEM...")
dem_normalized, dem_params = normalize_dem(dem_data, method='minmax')

print("\nReshaping DEM to 320×320...")
print(f"  Original shape: {dem_normalized.shape}")
zoom_factor_h = 320 / dem_normalized.shape[0] 
zoom_factor_w = 320 / dem_normalized.shape[1]  

# Resize using zoom
dem_normalized = zoom(dem_normalized, (zoom_factor_h, zoom_factor_w), order=1)  # order=1 for bilinear

print(f"  New shape: {dem_normalized.shape}")
print(f"  Value range: [{dem_normalized.min():.4f}, {dem_normalized.max():.4f}]")

dem_normalized, dem_params = normalize_dem(dem_normalized, method='minmax')
# Verify normalization
dem_norm_stats = get_raster_stats(dem_normalized, "DEM (Normalized)")
print_raster_stats(dem_norm_stats)

In [ ]:
# Normalize Infiltration
print("\nNormalizing Infiltration...")
infilt_normalized, infilt_params = normalize_infiltration(infilt_data)
# Verify normalization
infilt_norm_stats = get_raster_stats(infilt_normalized, "Infiltration (Normalized)")
print_raster_stats(infilt_norm_stats)

In [ ]:
# Normalize Landuse
print("\nNormalizing Landuse...")
landuse_normalized, landuse_params = normalize_landuse(landuse_data, method='minmax')
# Verify normalization
landuse_norm_stats = get_raster_stats(landuse_normalized, "Landuse (Normalized)")
print_raster_stats(landuse_norm_stats)

In [ ]:
# Normalize Drainage
drainage_norm = {}
for layer_name, layer_data in drainage_clean.items():
    # Drainage layers are already binary (0/1), just confirm range
    vmin, vmax = layer_data.min(), layer_data.max()
    if vmin == 0.0 and vmax == 1.0:
        # Already in [0, 1] — no normalization needed
        drainage_norm[layer_name] = layer_data.copy()
        print(f"  {layer_name}: Already [0, 1] ✓ (binary mask)")

In [ ]:
# Normalize Soil Types
soil_norm = {}
soil_norm_params = {}

for layer_name, layer_data in soil_clean.items():
    norm_data, params = normalize_dem(layer_data, method='minmax')  # reuse 
    soil_norm[layer_name] = norm_data
    soil_norm_params[layer_name] = params
    print(f"  {layer_name}: [{params['min']:.4f}, {params['max']:.4f}] → [0, 1] ✓")


In [ ]:
# Resize Drainage
TARGET_SHAPE = (320, 320)

drainage_resized = {}
for layer_name, layer_data in drainage_norm.items():
    zoom_h = TARGET_SHAPE[0] / layer_data.shape[0]
    zoom_w = TARGET_SHAPE[1] / layer_data.shape[1]
    resized = zoom(layer_data, (zoom_h, zoom_w), order=1)
    drainage_resized[layer_name] = resized
    print(f"  {layer_name}: {layer_data.shape} → {resized.shape} ✓")

In [ ]:
# Resize Drainage
soil_resized = {}
for layer_name, layer_data in soil_norm.items():
    zoom_h = TARGET_SHAPE[0] / layer_data.shape[0]
    zoom_w = TARGET_SHAPE[1] / layer_data.shape[1]
    resized = zoom(layer_data, (zoom_h, zoom_w), order=1)
    soil_resized[layer_name] = resized
    print(f"  {layer_name}: {layer_data.shape} → {resized.shape} ✓")

## Flood Maps (Ground Truths)

In [ ]:
print("\nFlood Susceptibility Class Definitions:")
for class_id, info in FLOOD_CLASSES.items():
    min_val, max_val = info['range']
    max_str = f"{max_val:.2f}" if max_val != float('inf') else "∞"
    print(f"  Class {class_id}: {info['name']:12s} - Range: [{min_val:.2f}, {max_str}) meters")


In [ ]:
flood_maps_categorized_patch, patch_metadata = categorize_all_flood_maps_patch(flood_maps, patch_size=PATCH_SIZE, stride=None, categorization_method='majority', verbose=True)

In [ ]:
reconstructed_maps = []
for patch_labels, metadata in zip(flood_maps_categorized_patch, patch_metadata):
    reconstructed = reconstruct_map_from_patches(patch_labels, metadata, method='nearest')
    reconstructed_maps.append(reconstructed)

fig_all = visualize_all_categorized_maps(reconstructed_maps, scenario_ids=list(range(1, 51)), figsize=(25, 20), ncols=5)
plt.show()

In [ ]:
display(flood_maps_categorized_patch)
# display(patch_metadata)

# Data Splitting Strategy

In [ ]:
print("\nScenario Split")
print("-" * 70)

import os
import re

def get_scenario_ids_from_dir(folder_path: str) -> List[int]:
    ids = []
    for fname in os.listdir(folder_path):
        if not os.path.isfile(os.path.join(folder_path, fname)):
            continue
        match = re.search(r'Scenario_(\d+)_mmhr', fname)
        if match:
            ids.append(int(match.group(1)))
    return sorted(ids)

train_scenario_ids = get_scenario_ids_from_dir(train_rs_path)
test_scenario_ids  = get_scenario_ids_from_dir(test_rs_path)

NUM_TRAIN = len(train_scenario_ids)
NUM_TEST  = len(test_scenario_ids)

print(f"Total scenarios: {NUM_TRAIN + NUM_TEST}")
print(f"Training scenarios: {NUM_TRAIN}")
print(f"Test scenarios: {NUM_TEST}")

train_rs_lookup = {sid: df for sid, df in zip(train_scenario_ids, train_rainfall_scenarios)}
test_rs_lookup  = {sid: df for sid, df in zip(test_scenario_ids,  test_rainfall_scenarios)}


def extract_rainfall_sequence(rainfall_df) -> np.ndarray:
    if 'intensity_mmhr' in rainfall_df.columns:
        return rainfall_df['intensity_mmhr'].values.astype(np.float32)


print("\nTraining Set Organization")
print("-" * 70)

train_data = {
    'scenario_ids':       train_scenario_ids,
    'num_scenarios':      NUM_TRAIN,
    'dem':                dem_normalized,
    'infiltration':       infilt_normalized,
    'landuse':            landuse_normalized,
    'drainage':           drainage_resized,  
    'soil':               soil_resized,      
    'rainfall_sequences': [],  
    'flood_patches':      [],  
}

for scenario_id in train_scenario_ids:
    flood_idx = scenario_id - 1
    seq = extract_rainfall_sequence(train_rs_lookup[scenario_id])
    train_data['rainfall_sequences'].append(seq)
    train_data['flood_patches'].append(flood_maps_categorized_patch[flood_idx])

print(f"Training Set:")
print(f"  Scenario IDs:         {train_data['scenario_ids']}")
print(f"  Number of scenarios:  {train_data['num_scenarios']}")
print(f"  --- Spatial Inputs (shared) ---")
print(f"  DEM:                  {train_data['dem'].shape}")
print(f"  Infiltration:         {train_data['infiltration'].shape}")
print(f"  Landuse:              {train_data['landuse'].shape}")
print(f"  Drainage layers ({len(train_data['drainage'])}):")
for name, arr in train_data['drainage'].items():
    print(f"    {name}: {arr.shape}")
print(f"  Soil layers ({len(train_data['soil'])}):")
for name, arr in train_data['soil'].items():
    print(f"    {name}: {arr.shape}")
print(f"  --- Per-Scenario Inputs ---")
print(f"  Rainfall sequences:   {len(train_data['rainfall_sequences'])} × {train_data['rainfall_sequences'][0].shape}")
print(f"  Flood patches:        {len(train_data['flood_patches'])} arrays × {len(train_data['flood_patches'][0])} patches")
print(f"  Total training patches: {len(train_data['flood_patches'][0]) * NUM_TRAIN:,}")

total_channels = 3 + len(drainage_resized) + len(soil_resized) 

print("\nTest Set Organization")
print("-" * 70)

test_data = {
    'scenario_ids':       test_scenario_ids,
    'num_scenarios':      NUM_TEST,
    'dem':                dem_normalized,
    'infiltration':       infilt_normalized,
    'landuse':            landuse_normalized,
    'drainage':           drainage_resized,
    'soil':               soil_resized,
    'rainfall_sequences': [],
    'flood_patches':      [],
}

for scenario_id in test_scenario_ids:
    flood_idx = scenario_id - 1
    seq = extract_rainfall_sequence(test_rs_lookup[scenario_id])
    test_data['rainfall_sequences'].append(seq)
    test_data['flood_patches'].append(flood_maps_categorized_patch[flood_idx])

print(f"Test Set:")
print(f"  Scenario IDs:         {test_data['scenario_ids']}")
print(f"  Number of scenarios:  {test_data['num_scenarios']}")
print(f"  --- Spatial Inputs (shared) ---")
print(f"  DEM:                  {test_data['dem'].shape}")
print(f"  Infiltration:         {test_data['infiltration'].shape}")
print(f"  Landuse:              {test_data['landuse'].shape}")
print(f"  Drainage layers ({len(test_data['drainage'])}):")
for name, arr in test_data['drainage'].items():
    print(f"    {name}: {arr.shape}")
print(f"  Soil layers ({len(test_data['soil'])}):")
for name, arr in test_data['soil'].items():
    print(f"    {name}: {arr.shape}")
print(f"  --- Per-Scenario Inputs ---")
print(f"  Rainfall sequences:   {len(test_data['rainfall_sequences'])} × {test_data['rainfall_sequences'][0].shape}")
print(f"  Flood patches:        {len(test_data['flood_patches'])} arrays × {len(test_data['flood_patches'][0])} patches")
print(f"  Total test patches:   {len(test_data['flood_patches'][0]) * NUM_TEST:,}")


print("\n" + "="*70)
print("DATASET SUMMARY")
print("="*70)
print(f"\n  Spatial channels per patch: {total_channels}")
print(f"  ViT input shape per patch:  ({total_channels}, PATCH_SIZE, PATCH_SIZE)")
print(f"  Rainfall input per sample:  (13,)")
print(f"  Total train patches:        {len(train_data['flood_patches'][0]) * NUM_TRAIN:,}")
print(f"  Total test patches:         {len(test_data['flood_patches'][0]) * NUM_TEST:,}")

# Patch Extraction for ViT

In [ ]:
print("\nPatch Extraction Strategy")
print("-" * 70)
print("Strategy: Non-overlapping patches")

num_patches_h = 320 // PATCH_SIZE
num_patches_w = 320 // PATCH_SIZE
total_patches = num_patches_h * num_patches_w

def extract_patches_2d(image: np.ndarray, patch_size: int) -> np.ndarray:
    h, w = image.shape
    assert h % patch_size == 0 and w % patch_size == 0, "Image size must be divisible by patch_size"
    
    num_patches_h = h // patch_size
    num_patches_w = w // patch_size
    
    # Reshape to extract patches
    patches = image.reshape(num_patches_h, patch_size, num_patches_w, patch_size)
    patches = patches.transpose(0, 2, 1, 3)  # (num_h, num_w, patch_size, patch_size)
    patches = patches.reshape(-1, patch_size, patch_size)  # (num_patches, patch_size, patch_size)
    
    return patches

In [ ]:
print("\nStacking Spatial Inputs")
print("-" * 70)

def extract_spatial_patches(dem: np.ndarray, infiltration: np.ndarray, landuse: np.ndarray, drainage: dict, soil: dict, patch_size: int) -> np.ndarray:
    # Extract patches per layer
    all_patches = [
        extract_patches_2d(dem, patch_size),          
        extract_patches_2d(infiltration, patch_size),  
        extract_patches_2d(landuse, patch_size),       
    ]
    # Add drainage layers
    for layer_data in drainage.values():
        all_patches.append(extract_patches_2d(layer_data, patch_size))
    # Add soil layers
    for layer_data in soil.values():
        all_patches.append(extract_patches_2d(layer_data, patch_size))
    # Stack as (num_patches, num_channels, patch_size, patch_size)
    stacked_patches = np.stack(all_patches, axis=1)
    return stacked_patches

# Channel inventory
channel_names = (['DEM', 'Infiltration', 'Landuse']+ [f'Drain_{k}' for k in drainage_resized.keys()]+ [f'Soil_{k}' for k in soil_resized.keys()])
num_channels = len(channel_names)

# Extract spatial patches 
spatial_patches = extract_spatial_patches( dem_normalized, infilt_normalized, landuse_normalized, drainage_resized, soil_resized, PATCH_SIZE)

print(f"Spatial patches extracted:")
print(f"  Shape: {spatial_patches.shape}")
print(f"  Expected: ({total_patches}, {num_channels}, {PATCH_SIZE}, {PATCH_SIZE})")
print(f"\n  Channels ({num_channels} total):")
for i, name in enumerate(channel_names):
    print(f"    [{i:2d}] {name}")

In [ ]:
print("\nCreating Complete Dataset Structure")
print("-" * 70)

def create_patch_dataset(spatial_patches, scenario_ids, rainfall_sequences, patch_labels):
    num_scenarios       = len(scenario_ids)
    patches_per_scenario = spatial_patches.shape[0]
    total_samples       = num_scenarios * patches_per_scenario

    rain_array = np.stack(rainfall_sequences, axis=0)  # (num_scenarios, 13)

    dataset = {
        'spatial_patches':    np.tile(spatial_patches[np.newaxis], (num_scenarios, 1, 1, 1, 1)),
        'rainfall_sequences': np.tile(rain_array[:, np.newaxis, :], (1, patches_per_scenario, 1)),
        'labels':             np.stack(patch_labels, axis=0),
        'scenario_ids':       scenario_ids,
        'num_scenarios':      num_scenarios,
        'patches_per_scenario': patches_per_scenario,
        'total_samples':      total_samples,
    }
    return dataset

print("\nCreating training dataset...")
train_dataset = create_patch_dataset(spatial_patches, train_scenario_ids,train_data['rainfall_sequences'], train_data['flood_patches'])
print(f"Training dataset:")
print(f"  Spatial patches:    {train_dataset['spatial_patches'].shape}")
print(f"  Rainfall sequences: {train_dataset['rainfall_sequences'].shape}")
print(f"  Labels:             {train_dataset['labels'].shape}")
print(f"  Total samples:      {train_dataset['total_samples']:,}")

print("\nCreating test dataset...")
test_dataset = create_patch_dataset(spatial_patches, test_scenario_ids, test_data['rainfall_sequences'], test_data['flood_patches'])
print(f"Test dataset:")
print(f"  Spatial patches:    {test_dataset['spatial_patches'].shape}")
print(f"  Rainfall sequences: {test_dataset['rainfall_sequences'].shape}")
print(f"  Labels:             {test_dataset['labels'].shape}")
print(f"  Total samples:      {test_dataset['total_samples']:,}")

In [ ]:
print("\n" + "="*70)
print("DATALOADER SETUP (with Temporal Sequences)")
print("="*70)

print("\nCreating PyTorch Dataset Class")
print("-" * 70)

class FloodPatchDataset(Dataset):
    def __init__(self, dataset_dict: Dict, transform=None):
        self.spatial_patches = dataset_dict['spatial_patches']      
        self.rainfall_sequences = dataset_dict['rainfall_sequences']
        self.labels = dataset_dict['labels']                         
        
        self.num_scenarios = dataset_dict['num_scenarios']
        self.patches_per_scenario = dataset_dict['patches_per_scenario']
        self.total_samples = dataset_dict['total_samples']
        self.scenario_ids = dataset_dict['scenario_ids']
        
        self.transform = transform
        
        # Flatten arrays for easier indexing
        # Spatial: (total_samples, 3, H, W)
        self.spatial_patches_flat = self.spatial_patches.reshape(
            self.total_samples, *self.spatial_patches.shape[2:]
        )
        
        # Rainfall sequences: (total_samples, 13)
        self.rainfall_sequences_flat = self.rainfall_sequences.reshape(
            self.total_samples, -1
        )
        
        # Labels: (total_samples,)
        self.labels_flat = self.labels.flatten()
        
        print(f"FloodPatchDataset initialized:")
        print(f"  Total samples: {self.total_samples:,}")
        print(f"  Scenarios: {self.num_scenarios}")
        print(f"  Patches per scenario: {self.patches_per_scenario}")
        print(f"  Spatial patch shape: {self.spatial_patches_flat.shape[1:]}")
        print(f"  Rainfall sequence shape: {self.rainfall_sequences_flat.shape[1:]}")
        
    def __len__(self) -> int:
        return self.total_samples
    
    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        # Get spatial patch (3, H, W)
        spatial_patch = self.spatial_patches_flat[idx]
        
        # Get rainfall sequence (13,)
        rainfall_seq = self.rainfall_sequences_flat[idx]
        
        # Get label (scalar)
        label = self.labels_flat[idx]
        
        # Convert to PyTorch tensors
        spatial_patch = torch.from_numpy(spatial_patch).float()
        rainfall_seq = torch.from_numpy(rainfall_seq).float() 
        label = torch.tensor(label, dtype=torch.long)
        
        # Apply transform
        if self.transform:
            spatial_patch = self.transform(spatial_patch)
        
        return spatial_patch, rainfall_seq, label
    
    def get_scenario_id(self, idx: int) -> int:
        scenario_idx = idx // self.patches_per_scenario
        return self.scenario_ids[scenario_idx]

# Create PyTorch datasets
train_pytorch_dataset = FloodPatchDataset(train_dataset)
test_pytorch_dataset = FloodPatchDataset(test_dataset)

print(f"\n PyTorch datasets created:")
print(f"  Training: {len(train_pytorch_dataset):,} samples")
print(f"  Test: {len(test_pytorch_dataset):,} samples")

# ViT Architecture

In [ ]:
print("\n" + "="*70)
print("VIT ARCHITECTURE DESIGN (WITH TEMPORAL RAINFALL)")
print("="*70)
 
from model_config import (update_dataset_info)
update_dataset_info(spatial_patches=spatial_patches, train_dataset=train_dataset, test_dataset=test_dataset, train_data=train_data, drainage_resized=drainage_resized, soil_resized=soil_resized,)

# Training Setup

In [ ]:
patches_per_scenario = 6400
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
scenario_to_samples = {}
for idx, scenario_id in enumerate(train_scenario_ids):
    start = idx * patches_per_scenario
    end = start + patches_per_scenario
    scenario_to_samples[scenario_id] = list(range(start, end))

fold_histories, best_fold = train_kfold(model_factory=create_small_model, full_dataset=train_pytorch_dataset, 
        scenario_to_samples=scenario_to_samples, batch_size=128, num_epochs=25, device=device, num_folds = 2)

# Testing

In [ ]:
print("\n" + "="*70)
print("TEST EVALUATION")
print("="*70)

print("\nLoading best model from training...")
best_fold_idx = best_fold  

model = create_small_model()
model_path = f'./checkpoints/fold_{best_fold_idx + 1}/best_model.pt'
model.load_state_dict(torch.load(model_path, map_location=device))
model.to(device)
model.eval()

print(f"✓ Loaded model from Fold {best_fold_idx + 1}")
print(f"  Model path: {model_path}")

In [ ]:
print("\nPreparing ground truth maps for test scenarios...")
ground_truth_maps = {}

for scenario_id in test_scenario_ids:
    # Get the index in test_dataset arrays
    scenario_idx = test_scenario_ids.index(scenario_id)
    # Extract ground truth from test_dataset
    ground_truth_patches = test_dataset['labels'][scenario_idx] 
    # Reconstruct to 320×320 map
    ground_truth_map = reconstruct_test_map_from_patches(ground_truth_patches, original_shape=(320,320), patch_size=PATCH_SIZE)
    ground_truth_maps[scenario_id] = ground_truth_map
    print(f"  Scenario {scenario_id}: Ground truth shape {ground_truth_map.shape}")
print(f"✓ Prepared {len(ground_truth_maps)} ground truth maps")


In [ ]:
print("\nEvaluating all test scenarios...")
all_results, all_predictions = evaluate_test_scenarios(model=model, patch_size=PATCH_SIZE, test_dataset=test_pytorch_dataset, test_scenario_ids=test_scenario_ids, ground_truth_maps=ground_truth_maps, device=device, save_dir='./results')

print("\nGenerating visualizations...")
results_dir = Path('./results')
results_dir.mkdir(parents=True, exist_ok=True)

for scenario_id in test_scenario_ids:
    print(f"\nVisualizing Scenario {scenario_id}...")
    
    # Confusion matrix
    plot_confusion_matrix(metrics=all_results[scenario_id], scenario_id=scenario_id, save_path=results_dir / f'confusion_matrix_scenario_{scenario_id}.png')
    # Ground truth vs prediction
    plot_scenario_comparison(scenario_id=scenario_id, ground_truth_map=ground_truth_maps[scenario_id], predicted_map=all_predictions[scenario_id],save_path=results_dir / f'comparison_scenario_{scenario_id}.png')
print("\n✓ All visualizations generated!")

print("\n" + "="*70)
print("DETAILED TEST RESULTS")
print("="*70)

for scenario_id in test_scenario_ids:
    metrics = all_results[scenario_id]
    
    print(f"\n{'='*70}")
    print(f"SCENARIO {scenario_id}")
    print(f"{'='*70}")
    print(f"Overall Accuracy: {metrics['accuracy']:.4f}")
    print(f"Mean IoU (mIoU):  {metrics['mean_iou']:.4f}")
    
    print("\nPer-Class Metrics:")
    print(f"{'Class':<10} {'IoU':<10} {'Precision':<12} {'Recall':<10} {'F1-Score':<10} {'Support':<10}")
    print("-" * 70)
    
    for class_id in range(5):
        iou = metrics['iou_per_class'][class_id]
        precision = metrics['precision_per_class'][class_id]
        recall = metrics['recall_per_class'][class_id]
        f1 = metrics['f1_per_class'][class_id]
        support = metrics['support_per_class'][class_id]
        
        print(f"{class_id:<10} {iou:<10.4f} {precision:<12.4f} {recall:<10.4f} {f1:<10.4f} {support:<10}")

print("\n" + "="*70)
print("TEST EVALUATION COMPLETE!")
print("="*70)